In [10]:
%reset -f
%run setup_notebook.py
from importlib import reload
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
import numpy as np
from like3.pixel_table import PixelTable
from like3.sourcelist import SourceModel

from like3 import main as _main; 
source_name = 'Mrk 421'
roi_sources = sm = SourceModel.from_fermi_catalog(
    source_name, #'v40', 
    cone_size=1.0,
    query='significance >= 25',
)
pt = PixelTable('files/kerr/toby_v4.fits', source_model=roi_sources)

if source_name in set(roi_sources.source_names):
    target_name = source_name
else:
    target_name = str(roi_sources.source_names[0])
    print(f" ROI centered on {source_name} ({target_name}) at {sm.selected_source.skydir.to_string()}")

Loaded Fermi 4FGL gll_psc_v40.fit: 7224 entries
Loaded pixel table from "files/kerr/toby_v4.fits":
            42 bands Band(0, 4): PSF0@1.33 GeV nside 64 occ 1.001 ... Band(3, 11): PSF3@74.99 GeV nside 2048 occ 0.000
            193,317,496 photons
            8,240,918 pixels, order ring
            
 ROI centered on Mrk 421 (J1104.4+3812) at 166.125 38.2064


In [11]:
reload(_main)
MultiBandLikelihood = _main.MultiBandLikelihood
mbl = MultiBandLikelihood(pt,sm)

In [24]:
from like3 import localization as _localization; reload(_localization)
Localization = _localization.Localization


Localizing source J1104.4+3812, tolerance=1.0e-03...
	    step      total     ra        dec       a         b         qual      
	       -      0.0000  166.1114   38.2069
	    0.0000    0.0000  166.1114   38.2069    0.0008    0.0008       0.0
	    0.0000    0.0000  166.1114   38.2069    0.0008    0.0008       0.0
	    0.0000    0.0000  166.1114   38.2069    0.0008    0.0008       0.0
TS change: 0.00


{'ra': 166.11139117952715,
 'dec': 38.20687228753675,
 'a': 0.0008181383656268449,
 'b': 0.0007921929188261963,
 'ang': -77.10270151539548,
 'qual': 0.009639223521878387,
 'sigma': 0.0008050611637077608}

In [ ]:

def localize(self, update=False, sigma=0.1, **kwargs):
    """Localize the currently selected source with a TS-map fit.

    Parameters
    ----------
    self : MultiBandLikelihood
        Likelihood object whose selected source will be localized.
    update : bool, optional
        If ``True``, keep the localized sky position applied by
        ``Localization.localize``. If ``False``, restore the original
        source position after the fit and return only the ellipse result.
    sigma : float, optional
        Initial localization scale in degrees. If the source already has
        an ``ellipse`` entry with a ``sigma`` value, that value is used
        instead.
    **kwargs
        Additional keyword arguments forwarded to ``Localization``.

    Returns
    -------
    dict
        Ellipse parameters returned by ``Localization.localize``. The
        result is also attached to ``source.ellipse``.

    Raises
    ------
    ValueError
        If no source is currently selected in the source model.
    """
   
    multilike = self

    class TSmap_function:

        def __init__(self, source) :
            self.source = source
            self.skydir = source.skydir
            self._llz = multilike.loglike(skydir=source.skydir)
 
        def __call__(self, skydir):
            """ TS map function: returns 2*(loglike(skydir) - loglike(nominal)) """
            return 2*(multilike.loglike(skydir=skydir) -self._llz)

    source = self.source_model.selected_source
    if source is None:
        raise ValueError("No source selected in the MultiBandLikelihood's source model")

    saved_skydir = source.skydir  

    # use existing ellipse sigma if available, otherwise default to 0.1 deg 
    if hasattr(source, 'ellipse'):
        sigma = source.ellipse.get('sigma', sigma)
    ellipse = Localization(TSmap_function(source), **kwargs).localize(sigma=sigma)
    if not update:
        # move only if requested, otherwise just return the ellipse parameters
        source.skydir = saved_skydir
    # attach result in any case
    source.ellipse = ellipse
    return ellipse
   

In [ ]:
 

ellipse = localize(mbl,update=True)
ellipse

In [11]:
from like3 import localization as _localization; reload(_localization)
Localization = _localization.Localization



class TSmap_function:
    
    def __init__(self, multilike) :
        if not isinstance(multilike, MultiBandLikelihood):
            raise ValueError("Expected a MultiBandLikelihood instance")
        # self.multilike = multilike
        self.source = multilike.source_model.selected_source
        if self.source is None:
            raise ValueError("No source selected in the MultiBandLikelihood's source model")
        self.skydir = self.source.skydir
        self._saved_skydir = self.skydir
        self._loglike = multilike.loglike

        self._llz = self._loglike(skydir=self.skydir)
        print(f"Initialized TSmap_function with source '{self.source.name}' at {self.skydir.to_string()}")

    def localize(self, update=False, **kwargs):
        """ Perform localization using the internal Localization instance. """
        return Localization(self, **kwargs).localize()

    def __call__(self, skydir):
        """ TS map function: returns 2*(loglike(skydir) - loglike(nominal)) """
        self.skydir = skydir
        return 2*(self._loglike(skydir=skydir) -self._llz)

tmf = TSmap_function(mbl)
tmf(SkyCoord(166.1138, 38.2088, unit='deg', frame='icrs'))

tmf.localize(update=True)

Initialized TSmap_function with source 'J1104.4+3812' at 166.114 38.2088
Localizing source J1104.4+3812, tolerance=1.0e-04...
	    step      total     ra        dec       a         b         qual      
	       -      0.0000  166.1138   38.2088
	    0.0059    0.0059  166.1072   38.2115    0.0000    0.0000      18.4
	    0.0052    0.0033  166.1104   38.2069    0.0006    0.0006     101.3
	    0.0006    0.0028  166.1112   38.2069    0.0008    0.0008      13.2
	    0.0001    0.0027  166.1113   38.2069    0.0008    0.0008       2.6
	    0.0000    0.0027  166.1114   38.2069    0.0008    0.0008       0.6
TS change: 11.45


True

In [ ]:
from like3 import localization as _localization; reload(_localization)
Localization = _localization.Localization

loc = Localization(tmf, verbose=False)
loc.localize()

In [ ]:

# if hasattr(loc, 'dir') and loc.dir is not None:
#     src.skydir = loc.dir.coord if hasattr(loc.dir, 'coord') else loc.dir

# def _to_skycoord(obj):
#     return obj.coord if hasattr(obj, 'coord') else obj

# def _scan_delta_ts_grid(loc, ra0, dec0, size_deg=0.4, nside=41):
#     dra = np.linspace(-size_deg, size_deg, nside)
#     ddec = np.linspace(-size_deg, size_deg, nside)
#     cosdec = max(abs(np.cos(np.radians(dec0))), 1e-6)
#     delta_ts = np.zeros((nside, nside), dtype=float)
#     for iy, dy in enumerate(ddec):
#         for ix, dx in enumerate(dra):
#             trial = SkyCoord(
#                 ra=ra0 + dx / cosdec,
#                 dec=dec0 + dy,
#                 unit='deg',
#                 frame='icrs',
#             )
#             delta_ts[iy, ix] = float(loc.TS(trial))
#     return dra, ddec, delta_ts

# center = _to_skycoord(src.skydir)
# ra0, dec0 = center.ra.deg, center.dec.deg

# # Auto-zoom until at least a few samples are above Delta TS = -25.
# size_deg = 0.4
# nside = 41
# for _ in range(6):
#     dra, ddec, delta_ts_raw = _scan_delta_ts_grid(loc, ra0, dec0, size_deg=size_deg, nside=nside)
#     core_mask = delta_ts_raw > -25.0
#     if np.count_nonzero(core_mask) >= 16:
#         break
#     size_deg *= 0.5
#     nside = min(161, nside + 20)

# # Keep only the requested Delta TS range: [-25, 0].
# delta_ts_grid = np.clip(delta_ts_raw, -25.0, 0.0)
# sqrt_minus_delta_ts = np.sqrt(-delta_ts_grid)

# xx, yy = np.meshgrid(dra, ddec)
# core_mask = np.isfinite(delta_ts_raw) & (delta_ts_raw > -25.0)
# if np.any(core_mask):
#     x_in = xx[core_mask]
#     y_in = yy[core_mask]
#     xspan = max(float(x_in.max() - x_in.min()), float(dra[1] - dra[0]))
#     yspan = max(float(y_in.max() - y_in.min()), float(ddec[1] - ddec[0]))
#     xpad = max(0.01, 0.25 * xspan)
#     ypad = max(0.01, 0.25 * yspan)
#     xlim = (float(x_in.min() - xpad), float(x_in.max() + xpad))
#     ylim = (float(y_in.min() - ypad), float(y_in.max() + ypad))
# else:
#     xlim = (float(dra.min()), float(dra.max()))
#     ylim = (float(ddec.min()), float(ddec.max()))

# fig, ax = plt.subplots(figsize=(6, 5))
# contours = ax.contourf(
#     dra,
#     ddec,
#     sqrt_minus_delta_ts,
#     levels=np.linspace(0.0, 5.0, 21),
#     vmin=0.0,
#     vmax=5.0,
#     cmap='jet_r',
#     extend='max',
# )
# ax.scatter([0], [0], c='white', s=60, marker='*', label='Current source position')

# if hasattr(loc, 'dir') and loc.dir is not None:
#     loc_sc = _to_skycoord(loc.dir)
#     ax.scatter(
#         [(loc_sc.ra.deg - ra0) * np.cos(np.radians(dec0))],
#         [loc_sc.dec.deg - dec0],
#         c='black',
#         s=45,
#         marker='x',
#         label='Localization center',
#     )

# ax.set_xlim(*xlim)
# ax.set_ylim(*ylim)
# ax.set_aspect('equal', adjustable='box')
# ax.set_xlabel('Delta RA * cos(dec) [deg]')
# ax.set_ylabel('Delta Dec [deg]')
# ax.set_title(r'$\sqrt{-\Delta TS}$ map around ' + target_name)
# ax.legend(loc='best')
# fig.colorbar(contours, ax=ax, label=r'$\sqrt{-\Delta TS}$')
# plt.show()

# print('Localization result:', loc)
# print('Localized position:', getattr(loc, 'dir', None))
# print(f'Grid size used: +/-{size_deg:.4f} deg, nside={nside}')
# print(f'Delta TS range before clipping: [{delta_ts_raw.min():.2f}, {delta_ts_raw.max():.2f}]')